# LA COPA DEL DECANO · Laboratorio de causalidad DGP v4 (30 participantes)

**Caso:** el Rector duda de la inversión en bienestar universitario. Primero observamos datos administrativos; luego incorporamos la encuesta; finalmente un A/B contrafactual simula el efecto de `do(B=4)` frente a `do(B=0)`.

El cuaderno está adaptado de las **especificaciones y ecuaciones** de `chapter-causalidad-master.zip`. Los datos exportados en el juego son sintéticos. **No prueban efectos de bienestar en personas reales.**

**Plan de 120 minutos:** 00–10 bienvenida · 10–30 juego · 30–55 Rector · 55–80 Encuesta/DAG · 80–105 A/B/DML · 105–120 Kahoot.

Este cuaderno es un recurso de exploración, **no 17 ejercicios obligatorios**.

## 1. Datos del Rector (Capa 1)

Descarga `sistema_academico_rector.csv` en la web del juego después de que el facilitador abra la fase Rector. En Colab o Jupyter puedes subirlo con el cuadro que se abre al ejecutar la siguiente celda.

In [ ]:
import io, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
try:
    from google.colab import files
    print("Sube el CSV del Rector")
    upload=files.upload()
    filename=next((n for n in upload if n.endswith("sistema_academico_rector.csv")),None)
    assert filename, "Selecciona sistema_academico_rector.csv"
    rector=pd.read_csv(io.BytesIO(upload[filename]))
except ImportError:
    rector=pd.read_csv("sistema_academico_rector.csv")
assert rector["nota_final"].between(0,50).all()
display(rector.head())
print("Filas:",len(rector),"Estudiantes:",rector.estudiante_id.nunique())

## 2. Trampa del Rector: predicción y variable omitida

El bienestar `B` puede parecer negativo si la muestra observacional confunde horas de bienestar con hábitos y características previas. Usar un coeficiente de OLS **no es una prueba de causalidad**.

Elegimos errores estándar robustos agrupados por estudiante: cada estudiante aporta varios exámenes, por lo que esas 450 observaciones **no son 450 personas independientes**.

In [ ]:
import statsmodels.formula.api as smf
reg=smf.ols("nota_final ~ horas_bienestar_sem + pga_previo + complejidad",data=rector).fit(
    cov_type="cluster",cov_kwds={"groups":rector.estudiante_id}
)
print(reg.summary().tables[1])
print("R²:",round(reg.rsquared,3))
print("Coeficiente predictivo/observacional del bienestar:",
      round(reg.params["horas_bienestar_sem"],3))
print("No interpretar este coeficiente como efecto causal identificado.")

**Microdecisión cerrada:** ¿el coeficiente de bienestar demuestra el efecto de reducir presupuesto? **No.** La selección de hábitos y la falta de identificación pueden crear sesgos.

## 3. Capa 2: encuesta de hábitos y predicción ML

Sube `encuesta_estudiantes_juego.csv`. Combina usando `estudiante_id` y `examen_nro` (ambas llaves), sin duplicar intentos.

In [ ]:
try:
    from google.colab import files
    print("Sube la encuesta")
    upload2=files.upload()
    filename=next((n for n in upload2 if n.endswith("encuesta_estudiantes_juego.csv")),None)
    assert filename, "Selecciona encuesta_estudiantes_juego.csv"
    survey=pd.read_csv(io.BytesIO(upload2[filename]))
except ImportError:
    survey=pd.read_csv("encuesta_estudiantes_juego.csv")
data=rector.merge(survey,on=["estudiante_id","examen_nro"],validate="one_to_one")
assert len(data)==len(rector),"Revisa que ambos archivos correspondan a la misma partida"
assert "fatiga_latente" not in data.columns
display(data.head())

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.model_selection import GroupShuffleSplit
features=["pga_previo","complejidad","horas_bienestar_sem","horas_estudio","horas_sueno"]
split=GroupShuffleSplit(n_splits=1,test_size=.30,random_state=2026)
train,test=next(split.split(data,groups=data.estudiante_id))
rf=RandomForestRegressor(n_estimators=60,min_samples_leaf=3,n_jobs=1,random_state=2026)
rf.fit(data.iloc[train][features],data.iloc[train].nota_final)
pred=rf.predict(data.iloc[test][features])
print("MAE en estudiantes reservados:",round(mean_absolute_error(data.iloc[test].nota_final,pred),2))
print("R² reservado:",round(r2_score(data.iloc[test].nota_final,pred),3))
display(pd.Series(rf.feature_importances_,index=features).sort_values(ascending=False).to_frame("Importancia predictiva"))
print("IMPORTANTE: importancia predictiva, incluso SHAP, no identifica un efecto.")

## 4. El DAG y los malos controles

`H=PGA previo`, `Z=complejidad`, `B=bienestar`, `X₁=estudio`, `S=sueño`, `U=fatiga oculta`, `Y=nota`.

El DGP del ZIP establece `H→B`, `H→Y`, `B→X₁`, `B→S`, `B→U`, `X₁→S`, `X₁→U`, `S→U` y `U→Y`. Las flechas son **supuestos de construcción**.

**Decisión:** para estimar el efecto TOTAL de B, ¿ajustamos indiscriminadamente X₁, S y U? **No.** Son descendientes potenciales del tratamiento. El arquetipo es una etiqueta UX, no una causa primitiva.

## 5. Experimento A/B Doctor Strange

Descarga `cohorte_ab_observada.csv` cuando el facilitador genere el A/B. **La asignación aleatoria es a nivel estudiante**, no por examen. Debemos agrupar por estudiante para estimar incertidumbre de forma válida.

In [ ]:
try:
    from google.colab import files
    print("Sube cohorte_ab_observada.csv")
    uploaded_ab=files.upload()
    filename=next((n for n in uploaded_ab if n.endswith("cohorte_ab_observada.csv")),None)
    assert filename,"Selecciona cohorte_ab_observada.csv"
    ab=pd.read_csv(io.BytesIO(uploaded_ab[filename]))
except ImportError:
    ab=pd.read_csv("cohorte_ab_observada.csv")
assert ab.groupby("estudiante_id")["tratamiento"].nunique().max()==1, "Tratamiento debe estar asignado por estudiante"
student_arms=ab.groupby(["estudiante_id","tratamiento"],as_index=False).nota_final.mean()
display(student_arms.groupby("tratamiento").nota_final.agg(["count","mean","std"]))
print("Observaciones:",len(ab),"Unidades aleatorizadas:",len(student_arms))

In [ ]:
from scipy import stats
treated=student_arms.loc[student_arms.tratamiento==1,"nota_final"].to_numpy()
control=student_arms.loc[student_arms.tratamiento==0,"nota_final"].to_numpy()
if len(treated)>1 and len(control)>1:
    ate=treated.mean()-control.mean()
    se=np.sqrt(treated.var(ddof=1)/len(treated)+control.var(ddof=1)/len(control))
    df=(se**4)/((treated.var(ddof=1)/len(treated))**2/(len(treated)-1)
        +(control.var(ddof=1)/len(control))**2/(len(control)-1))
    lower,upper=ate-stats.t.ppf(.975,df)*se,ate+stats.t.ppf(.975,df)*se
    print("Diferencia de medias aleatorizada:",round(ate,3))
    print("IC 95 % (Welch agrupado por estudiante):",tuple(round(x,3) for x in [lower,upper]))
else:
    print("Se necesitan por lo menos dos estudiantes en cada brazo para el intervalo.")
print("Los dos resultados potenciales de cada persona solo existen en el mundo sintético y en la verdad docente.")

## 6. DML: demostración opcional (no nuevo requisito del juego)

DML usa modelos auxiliares para estudiar `E[B|X]` y `E[Y|X]`, calcula residuos **fuera de muestra** y estima una asociación residual. Interpretarla como causal requiere un diseño válido, ignorabilidad condicional, consistencia y soporte común.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
X=data[["pga_previo","complejidad"]].to_numpy()
B=data.horas_bienestar_sem.to_numpy()
Y=data.nota_final.to_numpy()
rB=np.empty(len(data));rY=np.empty(len(data))
folds=GroupKFold(n_splits=min(3,data.estudiante_id.nunique()))
for tr,te in folds.split(X,Y,groups=data.estudiante_id):
    mb=RandomForestRegressor(n_estimators=30,min_samples_leaf=4,n_jobs=1,random_state=4)
    my=RandomForestRegressor(n_estimators=30,min_samples_leaf=4,n_jobs=1,random_state=9)
    mb.fit(X[tr],B[tr]);my.fit(X[tr],Y[tr])
    rB[te]=B[te]-mb.predict(X[te])
    rY[te]=Y[te]-my.predict(X[te])
denom=np.sum(rB*rB)
print("DML partialling-out:",round(np.sum(rB*rY)/denom,3) if denom>1e-9 else "Sin variación residual")
print("No confundir resultado ajustado con prueba automática de causalidad.")

## 7. Cierre: qué podemos concluir

- Una métrica predictiva no contesta `¿qué pasaría si hiciéramos do(B=4)?`.
- Ajustar mediadores puede bloquear el efecto total.
- Un A/B aleatoriza personas, no filas de exámenes.
- El DGP es sintético. Repetir otro torneo y cotejar la verdad docente prueba las ecuaciones, no efectos clínicos o educativos reales.

**Salida del laboratorio:** datos de Capa 1, Capa 2 y A/B observada, una comparación de estimaciones y una recomendación con supuestos explícitos.